<a href="https://colab.research.google.com/github/akbarruziev660-wq/Lesson/blob/main/%D0%A3%D1%80%D0%BE%D0%BA9_%D1%81%D0%B0%D0%BC%D0%BE%D1%81%D1%82%D0%BE%D1%8F%D1%82%D0%B5%D0%BB%D1%8C%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌲 Урок 9 — Самостоятельная работа: Случайный лес

### Как работать с этим ноутбуком
1. **Читай** текст в серых ячейках — это короткая теория.
2. **Запускай** ячейки с кодом по порядку (кнопка ▶ слева или `Shift+Enter`).
3. **Выполняй задания** 📝 — меняй код и смотри, что происходит.
4. **Отвечай словами** в ячейках ✍️ (дважды кликни, чтобы писать).
5. В конце — **финальное задание** и **проверь себя**.

> 🎯 **К концу урока ты сможешь:** объяснить, почему лес точнее одного дерева, построить `RandomForestClassifier`, прочитать важность признаков и сравнить модели честно (на test).

Работать не нужно устанавливать — всё уже есть в Colab. Поехали! 🚀

## Шаг 0 · Загружаем данные
Мы работаем с датасетом **Titanic** — кто выжил в катастрофе. Каждая строка — пассажир. Хотим предсказать `survived` (1 = выжил, 0 = нет).

*Просто запусти эту ячейку.*

In [ ]:
import seaborn as sns          # библиотека с готовыми датасетами
import pandas as pd            # таблицы (как Excel внутри Python)

df = sns.load_dataset('titanic')   # загружаем таблицу

# Выбираем понятные признаки
cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
data = df[cols + ['survived']].copy()

data['age'] = data['age'].fillna(data['age'].median())  # пропуски возраста -> медиана
data['sex'] = (data['sex'] == 'female').astype(int)     # пол в число: female=1, male=0

X = data[cols]         # признаки (по чему предсказываем)
y = data['survived']   # ответ (что предсказываем)

print('Строк:', len(X))
data.head()            # показать первые 5 строк

Строк: 891


,pclass,sex,age,sibsp,parch,fare,survived
0,3,0,22.0,1,0,7.2500,0
1,1,1,38.0,1,0,71.2833,1
2,3,1,26.0,0,0,7.9250,1
3,1,1,35.0,1,0,53.1000,1
4,3,0,35.0,0,0,8.0500,0


## Шаг 1 · Одно дерево «зазубривает»
Дерево решений задаёт вопросы «да/нет» (как игра «20 вопросов») и приходит к ответу.

Проблема: если дать ему волю, оно **запоминает** конкретных пассажиров вместо общего правила. Проверим это — сравним точность на **train** (данные, которые видело) и **test** (новые данные).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Делим данные: 80% на обучение, 20% на честную проверку
# random_state=42 -> у всех одинаковое разбиение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

tree = DecisionTreeClassifier(random_state=42)   # создаём дерево
tree.fit(X_train, y_train)                        # обучаем на train

# accuracy = доля верных ответов
acc_train = accuracy_score(y_train, tree.predict(X_train))
acc_test  = accuracy_score(y_test,  tree.predict(X_test))
print(f'Дерево — TRAIN: {acc_train:.0%}   TEST: {acc_test:.0%}')

Дерево — TRAIN: 98%   TEST: 76%


✍️ **Ответь словами.** На train дерево почти идеально, а на test заметно хуже. Почему так? (Подсказка: «зазубрить» ≠ «понять».)

*Твой ответ:*


…

## Шаг 2 · Сажаем лес
Идея: вместо одного дерева — **много разных деревьев**. Каждое учится на своей случайной части данных и признаков, поэтому все они разные. Чтобы предсказать — устраиваем **голосование**: что скажет большинство деревьев, то и ответ.

> 🗳️ Аналогия: голосование класса точнее одного мнения.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators=100 — сто деревьев в лесу
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)

acc_forest = accuracy_score(y_test, forest.predict(X_test))
print(f'Одно дерево, TEST: {acc_test:.0%}')
print(f'Лес (100),   TEST: {acc_forest:.0%}')

### 📝 Задание 1 — поэкспериментируй
Запусти ячейку ниже несколько раз, **меняя число деревьев** `n` (попробуй 1, 5, 10, 100, 300).
Заметь: точность сначала растёт, потом почти перестаёт меняться — это **плато**.

In [ ]:
n = 5   # 👈 МЕНЯЙ ЭТО ЧИСЛО (1, 5, 10, 100, 300) и запускай снова

f = RandomForestClassifier(n_estimators=n, random_state=42)
f.fit(X_train, y_train)
print(f'{n} деревьев -> точность на TEST: {accuracy_score(y_test, f.predict(X_test)):.1%}')

✍️ **Вывод.** Начиная примерно с какого числа деревьев точность почти перестаёт расти?

*Твой ответ:* …


## Шаг 3 · На что смотрит лес?
Лес умеет показать, какие признаки он использовал чаще и полезнее всего — `feature_importances_`.

In [ ]:
import matplotlib.pyplot as plt

# Собираем важности в удобную таблицу и сортируем
importances = pd.Series(forest.feature_importances_, index=X.columns)
importances = importances.sort_values()

importances.plot(kind='barh', color='#5B4FC4')   # горизонтальные столбики
plt.title('Важность признаков')
plt.show()

### 📝 Задание 2 — топ-3 признака
Выведи **три самых важных** признака. Попробуй сам, потом сравни с решением ниже.

In [ ]:
# Твой код здесь. Подсказка: у Series есть метод .sort_values(ascending=False) и .head(3)
top3 = importances.sort_values(ascending=False).head(3)
print(top3)

<details><summary>💡 Решение</summary>

```python
top3 = importances.sort_values(ascending=False).head(3)
print(top3)
```
</details>

> ❗ **Важно!** Высокая важность признака — это **не** доказательство причины. Это лишь «модель часто им пользовалась». Помни урок про корреляцию и причинность.

## Шаг 4 · Турнир: кто сильнее?
Сравним три модели на **одном и том же** test — только так сравнение честное.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

models = {
    'KNN':    KNeighborsClassifier(n_neighbors=5),
    'Дерево': DecisionTreeClassifier(random_state=42),
    'Лес':    RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    print(f'{name:7s}: {accuracy_score(y_test, m.predict(X_test)):.0%}')

## ✅ Проверь себя
Ответь для себя (можно вслух):
1. Почему лес обычно точнее одного дерева?
2. Что даёт разнообразие деревьев?
3. Если признак самый важный — значит ли это, что он **причина**?

<details><summary>Показать ответы</summary>

1. Много разных деревьев голосуют, случайные ошибки гасятся.
2. Разные деревья ошибаются по-разному — среднее устойчивее.
3. Нет! Важность ≠ причина.
</details>

## 🏁 Финальное задание (3 уровня)
**Базовый.** Обучи лес и выведи его точность на test.

**Средний.** Найди число деревьев, при котором точность выходит на плато (график из шага 1 задания).

**Продвинутый.** Включи `oob_score=True` — «бесплатную» проверку — и сравни её с точностью на test.

Пиши код в ячейке ниже 👇

In [ ]:
# Базовый уровень (стартер) — дополни при желании
final = RandomForestClassifier(n_estimators=100, random_state=42)
final.fit(X_train, y_train)
print(f'Точность на TEST: {accuracy_score(y_test, final.predict(X_test)):.1%}')

# Продвинутый уровень (раскомментируй):
# oob = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)
# oob.fit(X_train, y_train)
# print(f'OOB: {oob.oob_score_:.1%}  |  TEST: {accuracy_score(y_test, oob.predict(X_test)):.1%}')

---
### 🎉 Готово!
Ты научился строить случайный лес, читать важность признаков и честно сравнивать модели. На следующем уроке соберём всё в **конвейер (Pipeline)** и научимся оценивать модель ещё честнее.